In [2]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from datasets import load_dataset

dataset = load_dataset("Bitext/Bitext-customer-support-llm-chatbot-training-dataset")
X = dataset['train']

In [3]:
corpus = X['instruction'][:2000]
corpus

['question about cancelling order {{Order Number}}',
 'i have a question about cancelling oorder {{Order Number}}',
 'i need help cancelling puchase {{Order Number}}',
 'I need to cancel purchase {{Order Number}}',
 'I cannot afford this order, cancel purchase {{Order Number}}',
 'can you help me cancel order {{Order Number}}?',
 'I can no longer afford order {{Order Number}}, cancel it',
 'I am trying to cancel purchase {{Order Number}}',
 'I have got to cancel purchase {{Order Number}}',
 'i need help canceling purchase {{Order Number}}',
 'i dont know what to do to cancel order {{Order Number}}',
 'I have a problem with cancelling purchase {{Order Number}}',
 'problem with canceling purchase {{Order Number}}',
 'can you help me canceling order {{Order Number}}?',
 'help  me canceling purchase {{Order Number}}',
 'can you help me cancel purchase {{Order Number}}?',
 'what do I have to do to cancel purchase {{Order Number}}?',
 'I bought the same item twice, cancel order {{Order Numbe

# Generate Embeddings and train model on embeddings.

In [4]:
import sys
sys.path.append("../src")
from embeddings import embed_tickets
from kmeans import CustomKmeansPlusPlus

X_embed = embed_tickets(corpus)

model = CustomKmeansPlusPlus(k=8, max_iters=100, random_state=42)
model.fit(X_embed)
X_embed.shape

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/143 [00:00<?, ?it/s]

(2000, 384)

# Calculate new Silhouette Score

In [10]:
from sklearn.metrics import silhouette_score

k_values = [3, 5, 8, 12, 16, 20, 25]

for k in k_values:
    model = CustomKmeansPlusPlus(k=k, max_iters=50, random_state=42)
    model.fit(X_embed)
    # inertias.append(model.inertia_)
    tfidf_silhouette_score = silhouette_score(X_embed, model.labels)
    print(tfidf_silhouette_score)

print(f"Layer 2 (Embeddings) Silhouette Score: {tfidf_silhouette_score:.4f}")
print("Compare this to your Layer 1 TF-IDF score of 0.1765!")

0.2555494010448456
0.2711915373802185
0.2508499026298523
0.25515010952949524
0.216274693608284
0.21596388518810272
0.2249062955379486
Layer 2 (Embeddings) Silhouette Score: 0.2249
Compare this to your Layer 1 TF-IDF score of 0.1765!


# Visualising Dense space

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
X_pca_dense = pca.fit_transform(X_embed)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(X_pca_dense[:, 0], X_pca_dense[:, 1], 
                      c=model.labels, cmap='tab10', alpha=0.6, s=15)

plt.title("PCA Projection of Semantic Embedding Clusters")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.colorbar(scatter, label="Cluster Label")
plt.show()

# Exporting sklearn's KMeans model with entire 27000 corpus
## _This is an Anchor file. Exporting "model" (CustomKMeansPlusPlus) or "km" (sklearn's KMeans model) decides performance_

In [ ]:
corpus = X['instruction']
len(corpus)

In [ ]:
from embeddings import embed_tickets

X_embed = embed_tickets(corpus)
X_embed.shape

In [ ]:
from sklearn.cluster import KMeans

km = KMeans(
    n_init=10,
    n_clusters=12,
    tol=1e-4,
    max_iter=300,
    init='k-means++',
    random_state=42)

km.fit(X_embed)

In [ ]:
from vector_store import TicketIndexer

indexer = TicketIndexer(vector_dim=X_embed.shape[1], use_hnsw=True)
indexer.build_index(X_embed)

In [ ]:
import pickle

# Pickle dump model
"""
Change the model to either "model" or "km" in the following pickle.dump line
"""
with open('../data/processed/kmeans_model.pkl', 'wb') as file:
    pickle.dump(km, file)

print("Model file dumped Successfully ✅")

# Numpy save corpus
np.save('../data/processed/embeddings.npy', X_embed)
print("Embeddings saved Successfully ✅")


# Pickle dump corpus
with open('../data/processed/corpus.pkl', 'wb') as file:
    pickle.dump(corpus, file)

print("Corpus file dumped Successfully ✅")